In [ ]:
import os
import shutil
from pathlib import Path

# ==================== 配置开关 ====================
FORCE_RECOMPUTE_FEATURES = True
FORCE_RETRAIN = True
FORCE_RECREATE_FIGS = True

# ==================== 基础配置 ====================
USE_TEST = 0
VERSION_DIR = Path.cwd()

FIG_DIR = VERSION_DIR / "fig"
MODEL_DIR = VERSION_DIR / "model"
FEATURE_DIR = VERSION_DIR / "featured_data"

ROOT_DIR = VERSION_DIR.parent
DATA_DIR = ROOT_DIR / "data"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

for dir_path in [FIG_DIR, MODEL_DIR, FEATURE_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# ==================== WSL mixed-disk scratch strategy ====================
# Strategy:
# 1. Spark local temp uses both C and D to spread shuffle / spill writes
# 2. Java tmp and text stage files stay on D only
# 3. Final outputs stay in the project output directories

SPARK_LOCAL_DIR_CANDIDATES = [
    Path("./scratch/c/spark_tmp_local"),
    Path("./scratch/d/spark_tmp_local"),
]

JAVA_TMP_CANDIDATES = [
    Path("./scratch/d/spark_tmp_java"),
    VERSION_DIR / "_spark_tmp_java",
]

TEXT_STAGE_CANDIDATES = [
    Path("./scratch/d/spark_tmp_text"),
    VERSION_DIR / "_spark_tmp_text",
]


def choose_writable_path(candidates):
    last_error = None

    for path in candidates:
        try:
            path = Path(path)
            path.mkdir(parents=True, exist_ok=True)
            probe = path / ".write_test"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return path
        except Exception as e:
            last_error = e
            print(f"[scratch] unavailable: {path} -> {e}")

    raise RuntimeError(f"no writable path found, last error: {last_error}")


def choose_multiple_writable_paths(candidates):
    writable = []

    for path in candidates:
        try:
            path = Path(path)
            path.mkdir(parents=True, exist_ok=True)
            probe = path / ".write_test"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            writable.append(path)
        except Exception as e:
            print(f"[spark_local] unavailable: {path} -> {e}")

    if not writable:
        fallback = VERSION_DIR / "_spark_tmp_local"
        fallback.mkdir(parents=True, exist_ok=True)
        writable = [fallback]

    return writable


SPARK_LOCAL_DIRS = choose_multiple_writable_paths(SPARK_LOCAL_DIR_CANDIDATES)
SPARK_LOCAL_DIRS_VALUE = ",".join(str(p) for p in SPARK_LOCAL_DIRS)

JAVA_TMP_DIR = choose_writable_path(JAVA_TMP_CANDIDATES)
TEXT_STAGE_ROOT = choose_writable_path(TEXT_STAGE_CANDIDATES)

SHARED_WORDS_DIR = TEXT_STAGE_ROOT / "shared_words"
METHOD_STAGE_ROOT = TEXT_STAGE_ROOT / "stage_parquet"
VOCAB_EXPORT_DIR = TEXT_STAGE_ROOT / "debug_vocab"


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def clear_dir_contents(path):
    path = Path(path)
    if not path.exists():
        return

    for child in path.iterdir():
        try:
            if child.is_dir():
                shutil.rmtree(child)
            else:
                child.unlink()
        except Exception as e:
            print(f"[warn] cleanup failed: {child} -> {e}")


def apply_scratch_env():
    for path in [
        JAVA_TMP_DIR,
        TEXT_STAGE_ROOT,
        SHARED_WORDS_DIR,
        METHOD_STAGE_ROOT,
        VOCAB_EXPORT_DIR,
    ]:
        ensure_dir(path)

    for path in SPARK_LOCAL_DIRS:
        ensure_dir(path)

    os.environ["SPARK_LOCAL_DIRS"] = SPARK_LOCAL_DIRS_VALUE
    os.environ["TEMP"] = str(JAVA_TMP_DIR)
    os.environ["TMP"] = str(JAVA_TMP_DIR)


print(f"VERSION_DIR: {VERSION_DIR}")
print(f"FIG_DIR: {FIG_DIR}")
print(f"MODEL_DIR: {MODEL_DIR}")
print(f"FEATURE_DIR: {FEATURE_DIR}")
print(f"TRAIN_PATH: {TRAIN_PATH}")
print(f"TEST_PATH: {TEST_PATH}")

print("\n=== mixed scratch strategy ===")
print("SPARK_LOCAL_DIRS:")
for p in SPARK_LOCAL_DIRS:
    print(f" - {p}")
print(f"SPARK_LOCAL_DIRS_VALUE: {SPARK_LOCAL_DIRS_VALUE}")
print(f"JAVA_TMP_DIR: {JAVA_TMP_DIR}")
print(f"TEXT_STAGE_ROOT: {TEXT_STAGE_ROOT}")
print(f"SHARED_WORDS_DIR: {SHARED_WORDS_DIR}")
print(f"METHOD_STAGE_ROOT: {METHOD_STAGE_ROOT}")
print(f"VOCAB_EXPORT_DIR: {VOCAB_EXPORT_DIR}")
print(f"FORCE_RECOMPUTE_FEATURES: {FORCE_RECOMPUTE_FEATURES}")

In [ ]:
# ==================== Spark init ====================
from pyspark.sql import SparkSession


def build_spark_session():
    apply_scratch_env()

    java_tmp_str = str(JAVA_TMP_DIR).replace("\\", "/")
    java_tmp_opt = f'-Djava.io.tmpdir="{java_tmp_str}"'

    spark_session = (
        SparkSession.builder
        .appName("V3_Survival_Mode_MixedDisk")
        .master("local[16]")
        .config("spark.driver.memory", "18g")
        .config("spark.executor.memory", "8g")
        .config("spark.shuffle.compress", "true")
        .config("spark.shuffle.spill.compress", "true")
        .config("spark.io.compression.codec", "lz4")
        .config("spark.sql.shuffle.partitions", "64")
        .config("spark.default.parallelism", "64")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .config("spark.local.dir", SPARK_LOCAL_DIRS_VALUE)
        .config("spark.driver.extraJavaOptions", java_tmp_opt)
        .config("spark.executor.extraJavaOptions", java_tmp_opt)
        .getOrCreate()
    )

    return spark_session


try:
    spark.stop()
except Exception:
    pass

spark = build_spark_session()

EXPECTED_TRAIN_ROWS = 5_173_005

final_test_path = FEATURE_DIR / "final_test.parquet"
if final_test_path.exists():
    EXPECTED_TEST_ROWS = spark.read.parquet(str(final_test_path)).count()
else:
    EXPECTED_TEST_ROWS = None
    print(f"[warn] final_test.parquet not found: {final_test_path}")

print("Spark init complete")
print(f"SPARK_LOCAL_DIRS_VALUE: {SPARK_LOCAL_DIRS_VALUE}")
print("SPARK_LOCAL_DIRS:")
for p in SPARK_LOCAL_DIRS:
    print(f" - {p}")
print(f"JAVA_TMP_DIR: {JAVA_TMP_DIR}")
print(f"TEXT_STAGE_ROOT: {TEXT_STAGE_ROOT}")
print(f"spark.master: {spark.sparkContext.master}")
print(f"spark.local.dir: {spark.conf.get('spark.local.dir')}")
print(f"spark.sql.shuffle.partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"spark.default.parallelism: {spark.conf.get('spark.default.parallelism')}")
print(f"spark.sql.adaptive.enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"spark.sql.adaptive.coalescePartitions.enabled: {spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled')}")
print(f"EXPECTED_TRAIN_ROWS: {EXPECTED_TRAIN_ROWS}")
print(f"EXPECTED_TEST_ROWS: {EXPECTED_TEST_ROWS}")

In [ ]:
from pathlib import Path
import gc

from pyspark import StorageLevel
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array, array_to_vector


class TextFeatures:
    """
    Layer2 text feature builder.
    Shared words stay on disk for reuse.
    Title/comment/improved middle stages stay in memory as much as possible.
    """

    def __init__(
        self,
        spark,
        raw_data_path,
        first_level_data_path,
        w2v_model_path,
        tfidf_model_path,
        output_path,
        method="tfidf_weighted",
        force_recompute=False,
    ):
        self.spark = spark
        self.raw_data_path = Path(raw_data_path)
        self.first_level_data_path = Path(first_level_data_path)
        self.w2v_model_path = Path(w2v_model_path)
        self.tfidf_model_path = Path(tfidf_model_path)
        self.output_path = Path(output_path) if output_path else None
        self.method = method
        self.force_recompute = force_recompute

        self.output_dir = self.output_path.parent if self.output_path else Path.cwd()
        self.expected_count = getattr(self, "expected_count", None)
        self.dataset_name = getattr(self, "dataset_name", None)
        self.final_improved_path = None
        self.new_cols = []

        self.vector_size = None
        self.word_vectors = {}
        self.num_partitions = int(self.spark.conf.get("spark.sql.shuffle.partitions", "64"))

    # -------------------- path helpers --------------------
    def _get_dataset_name(self):
        if getattr(self, "dataset_name", None):
            return self.dataset_name
        path_str = str(self.first_level_data_path).lower()
        return "test" if "test" in path_str else "train"

    def _get_expected_count(self):
        if getattr(self, "expected_count", None) is not None:
            return self.expected_count
        if self._get_dataset_name() == "train":
            return globals().get("EXPECTED_TRAIN_ROWS", 5_173_005)
        return globals().get("EXPECTED_TEST_ROWS", None)

    def _resolve_output_dir(self):
        return Path(getattr(self, "output_dir", self.output_path.parent if self.output_path else Path.cwd()))

    def _resolve_final_output_path(self):
        if self.final_improved_path:
            return Path(self.final_improved_path)
        file_name = "final_improved_test.parquet" if self._get_dataset_name() == "test" else "final_improved_train.parquet"
        return self._resolve_output_dir() / file_name

    def _resolve_version_dir(self):
        return self.first_level_data_path.parent.parent

    def _resolve_shared_words_path(self):
        shared_dir = Path(globals().get("SHARED_WORDS_DIR", self._resolve_version_dir() / "_shared_text_tmp"))
        shared_dir.mkdir(parents=True, exist_ok=True)
        file_name = "train_words.parquet" if self._get_dataset_name() == "train" else "test_words.parquet"
        return shared_dir / file_name

    def _persist_df(self, df, name):
        df = df.persist(StorageLevel.MEMORY_AND_DISK)
        print(f"[persist] {name}")
        return df


    # -------------------- integrity checks --------------------
    def _check_row_count(self, df, name, expected_count=None):
        count = df.count()
        print(f"[row_count] {name}: {count}")
        if expected_count is not None and count != expected_count:
            raise ValueError(f"{name} row count mismatch: expected {expected_count}, got {count}")
        return count

    def _check_unique_id(self, df, name):
        row = df.agg(
            F.count("*").alias("total_count"),
            F.countDistinct("id").alias("unique_count"),
        ).collect()[0]

        total_count = int(row["total_count"])
        unique_count = int(row["unique_count"])

        print(f"[unique_id] {name}: total={total_count}, unique_id={unique_count}")
        if total_count != unique_count:
            raise ValueError(f"{name} contains duplicate ids")
        return total_count, unique_count

    def _check_id_duplicates(self, df, name):
        dup_count = df.groupBy("id").count().filter(F.col("count") > 1).count()
        print(f"[duplicate_id] {name}: {dup_count}")
        if dup_count > 0:
            raise ValueError(f"{name} contains duplicate ids")

    def _check_key_columns_pattern(self, df, name):
        if "prod_id" in df.columns:
            bad_prod = df.filter(F.col("prod_id").isNull() | (~F.col("prod_id").rlike(r"^a_"))).count()
            print(f"[key_check] {name} bad prod_id rows: {bad_prod}")
            if bad_prod > 0:
                raise ValueError(f"{name} has invalid prod_id values")

        if "user_id" in df.columns:
            bad_user = df.filter(F.col("user_id").isNull() | (~F.col("user_id").rlike(r"^u_"))).count()
            print(f"[key_check] {name} bad user_id rows: {bad_user}")
            if bad_user > 0:
                raise ValueError(f"{name} has invalid user_id values")

        if "rating" in df.columns:
            bad_rating = df.filter(
                F.col("rating").isNotNull()
                & (
                    F.col("rating").cast("int").isNull()
                    | (~F.col("rating").cast("int").isin([1, 2, 3, 4, 5]))
                )
            ).count()
            print(f"[key_check] {name} bad rating rows: {bad_rating}")
            if bad_rating > 0:
                raise ValueError(f"{name} has rating values outside 1..5")

    def _resolve_integrity_mode(self, name, mode="auto"):
        if mode in {"full", "light"}:
            return mode
        if mode != "auto":
            raise ValueError(f"Unknown integrity mode: {mode}")

        full_names = (
            "-raw-words-id",
            "-improve-text-output",
            "-original-before-merge",
            "-after-merge",
            "-verify-input",
        )

        light_names = (
            "-layer1-final",
            "-shared-words-memory",
            "-title-stage-input",
            "-title-stage-output",
            "-comment-stage-input",
            "-comment-stage-output",
            "-assemble-title-input",
            "-assemble-comment-input",
            "-save-improved-features",
            "-improved-text-before-merge",
        )

        if any(marker in name for marker in full_names):
            return "full"
        if any(marker in name for marker in light_names):
            return "light"

        return "full"

    def _full_integrity_check(self, df, name, expected_count=None, check_patterns=True):
        total_count, unique_count = self._check_unique_id(df, name)

        if expected_count is not None and total_count != expected_count:
            raise ValueError(f"{name} row count mismatch: expected {expected_count}, got {total_count}")

        self._check_id_duplicates(df, name)

        if check_patterns:
            self._check_key_columns_pattern(df, name)

        return total_count, unique_count

    def _light_row_check(self, df, name, expected_count=None):
        count = self._check_row_count(df, name, expected_count=expected_count)
        return count, None

    def _assert_integrity(self, df, name, expected_count=None, check_patterns=True, mode="auto"):
        resolved_mode = self._resolve_integrity_mode(name, mode)
        if resolved_mode == "light":
            return self._light_row_check(df, name, expected_count=expected_count)
        return self._full_integrity_check(
            df,
            name,
            expected_count=expected_count,
            check_patterns=check_patterns,
        )

    def _safe_sample_compare(self, base_df, target_df, sample_ids, cols):
        valid_cols = [c for c in cols if c in base_df.columns and c in target_df.columns]
        if not valid_cols:
            print("[sample_compare] no shared columns to compare")
            return

        print("[sample_compare] base rows")
        base_df.filter(F.col("id").isin(sample_ids)).select("id", *valid_cols).orderBy("id").show(truncate=False)

        print("[sample_compare] merged rows")
        target_df.filter(F.col("id").isin(sample_ids)).select("id", *valid_cols).orderBy("id").show(truncate=False)

    # -------------------- cleanup --------------------
    def cleanup_stage_resources(self, *dfs):
        for df in dfs:
            if df is None:
                continue
            try:
                df.unpersist(blocking=False)
            except Exception:
                pass

        try:
            self.spark.catalog.clearCache()
        except Exception:
            pass

        gc.collect()

    # -------------------- data loading --------------------
    def load_first_level_features(self):
        dataset_name = self._get_dataset_name()
        expected_count = self._get_expected_count()

        df = self.spark.read.parquet(str(self.first_level_data_path))
        self._assert_integrity(
            df,
            f"{dataset_name}-layer1-final",
            expected_count=expected_count,
            check_patterns=False,
            mode="light",
        )
        return df

    def prepare_words_stage(self):
        words_path = self._resolve_shared_words_path()
        if words_path.exists() and not self.force_recompute:
            print(f">>> reuse shared words stage: {words_path}")
            return words_path

        print(">>> build tokenized words from raw csv")
        df_raw = self.spark.read.csv(
            str(self.raw_data_path),
            header=True,
            inferSchema=True,
            multiLine=True,
            escape='"',
            quote='"',
        )

        from pyspark.ml.feature import Tokenizer, StopWordsRemover

        df_raw = df_raw.withColumn(
            "cleaned_title",
            F.when(F.col("title").isNull(), F.lit("")).otherwise(F.col("title"))
        ).withColumn(
            "cleaned_comment",
            F.when(F.col("comment").isNull(), F.lit("")).otherwise(F.col("comment"))
        )

        df_raw = Tokenizer(inputCol="cleaned_title", outputCol="title_words").transform(df_raw)
        df_raw = Tokenizer(inputCol="cleaned_comment", outputCol="comment_words").transform(df_raw)
        df_raw = StopWordsRemover(inputCol="title_words", outputCol="title_words_filtered").transform(df_raw)
        df_raw = StopWordsRemover(inputCol="comment_words", outputCol="comment_words_filtered").transform(df_raw)

        words_df = df_raw.select("id", "title_words_filtered", "comment_words_filtered")
        self._assert_integrity(
            words_df.select("id"),
            f"{self._get_dataset_name()}-raw-words-id",
            expected_count=self._get_expected_count(),
            check_patterns=False,
            mode="full",
        )

        words_df.repartition(16).write.mode("overwrite").parquet(str(words_path))
        print(f"[ok] shared words saved to: {words_path}")

        self.cleanup_stage_resources(df_raw, words_df)
        return words_path

    def load_word2vec_model(self):
        from pyspark.ml.feature import Word2VecModel
        title_model = Word2VecModel.load(str(self.w2v_model_path / "title_w2v.model"))
        comment_model = Word2VecModel.load(str(self.w2v_model_path / "comment_w2v.model"))
        return title_model, comment_model

    def load_tfidf_models(self):
        from pyspark.ml.feature import CountVectorizerModel, IDFModel
        title_cv = CountVectorizerModel.load(str(self.tfidf_model_path / "title_cv.model"))
        title_idf = IDFModel.load(str(self.tfidf_model_path / "title_idf.model"))
        comment_cv = CountVectorizerModel.load(str(self.tfidf_model_path / "comment_cv.model"))
        comment_idf = IDFModel.load(str(self.tfidf_model_path / "comment_idf.model"))
        return (title_cv, title_idf), (comment_cv, comment_idf)

    # -------------------- spark-side feature builders --------------------
    def _build_vocab_df(self, cv_model):
        rows = [(word, int(idx)) for idx, word in enumerate(cv_model.vocabulary)]
        return self.spark.createDataFrame(rows, ["word", "tfidf_idx"])

    def _build_idf_df(self, idf_model):
        idf_values = idf_model.idf.toArray().tolist()
        rows = [(int(idx), float(val)) for idx, val in enumerate(idf_values)]
        return self.spark.createDataFrame(rows, ["tfidf_idx", "idf"])

    def _build_word_vector_df(self, model):
        return model.getVectors().select("word", F.col("vector").alias("word_vector"))

    def _zero_vector(self):
        return array_to_vector(F.expr(f"array_repeat(0D, {self.vector_size})"))

    def _weighted_pool_single(self, df, words_col, out_col, vocab_df, idf_df, vector_df):
        base_ids = df.select("id").distinct()

        token_df = (
            df.select("id", F.explode_outer(F.col(words_col)).alias("word"))
            .filter(F.col("word").isNotNull())
        )

        tf_df = token_df.groupBy("id", "word").agg(F.count(F.lit(1)).alias("tf"))

        weighted_df = (
            tf_df.join(vocab_df, on="word", how="inner")
            .join(idf_df, on="tfidf_idx", how="inner")
            .withColumn("weight", F.col("tf").cast("double") * F.col("idf"))
            .filter(F.col("weight") > 0)
            .join(vector_df, on="word", how="inner")
            .select(
                "id",
                F.col("weight").alias("weight"),
                vector_to_array("word_vector").alias("vec_arr"),
            )
            .repartition(self.num_partitions, "id")
        )

        sum_exprs = [
            F.sum(F.col("vec_arr")[i] * F.col("weight")).alias(f"s_{i}")
            for i in range(self.vector_size)
        ]

        agg_df = weighted_df.groupBy("id").agg(
            F.sum("weight").alias("total_weight"),
            *sum_exprs
        )

        value_cols = [
            F.when(F.col("total_weight") > 0, F.col(f"s_{i}") / F.col("total_weight")).otherwise(F.lit(0.0))
            for i in range(self.vector_size)
        ]

        pooled_df = agg_df.select("id", array_to_vector(F.array(*value_cols)).alias(out_col))

        result_df = (
            base_ids.join(pooled_df, on="id", how="left")
            .withColumn(out_col, F.when(F.col(out_col).isNull(), self._zero_vector()).otherwise(F.col(out_col)))
            .select("id", out_col)
        )

        self.cleanup_stage_resources(token_df, tf_df, weighted_df, agg_df, pooled_df)
        return result_df

    def _max_pool_single(self, df, words_col, out_col, vector_df):
        base_ids = df.select("id").distinct()

        token_df = (
            df.select("id", F.explode_outer(F.col(words_col)).alias("word"))
            .filter(F.col("word").isNotNull())
            .join(vector_df, on="word", how="inner")
            .select("id", vector_to_array("word_vector").alias("vec_arr"))
            .repartition(self.num_partitions, "id")
        )

        max_exprs = [
            F.max(F.col("vec_arr")[i]).alias(f"m_{i}")
            for i in range(self.vector_size)
        ]

        agg_df = token_df.groupBy("id").agg(*max_exprs)

        arr_cols = [
            F.coalesce(F.col(f"m_{i}").cast("double"), F.lit(0.0))
            for i in range(self.vector_size)
        ]

        pooled_df = agg_df.select("id", array_to_vector(F.array(*arr_cols)).alias(out_col))

        result_df = (
            base_ids.join(pooled_df, on="id", how="left")
            .withColumn(out_col, F.when(F.col(out_col).isNull(), self._zero_vector()).otherwise(F.col(out_col)))
            .select("id", out_col)
        )

        self.cleanup_stage_resources(token_df, agg_df, pooled_df)
        return result_df

    # -------------------- in-memory staged builders --------------------
    def build_title_feature_only(self, words_df):
        dataset_name = self._get_dataset_name()

        title_words_df = words_df.select("id", "title_words_filtered")
        self._assert_integrity(
            title_words_df.select("id"),
            f"{dataset_name}-title-stage-input",
            expected_count=self._get_expected_count(),
            check_patterns=False,
            mode="light",
        )

        if self.method == "tfidf_weighted":
            result_df = self._weighted_pool_single(
                title_words_df,
                words_col="title_words_filtered",
                out_col="title_w2v_tfidf",
                vocab_df=self.title_vocab_df,
                idf_df=self.title_idf_df,
                vector_df=self.title_vector_df,
            )
        else:
            result_df = self._max_pool_single(
                title_words_df,
                words_col="title_words_filtered",
                out_col="title_w2v_max",
                vector_df=self.title_vector_df,
            )

        self._assert_integrity(
            result_df.select("id"),
            f"{dataset_name}-title-stage-output",
            expected_count=self._get_expected_count(),
            check_patterns=False,
            mode="light",
        )

        result_df = self._persist_df(result_df, f"{dataset_name}-title-stage-output-memory")
        print("[ok] title stage kept in memory")
        return result_df

    def build_comment_feature_only(self, words_df):
        dataset_name = self._get_dataset_name()

        comment_words_df = words_df.select("id", "comment_words_filtered")
        self._assert_integrity(
            comment_words_df.select("id"),
            f"{dataset_name}-comment-stage-input",
            expected_count=self._get_expected_count(),
            check_patterns=False,
            mode="light",
        )

        if self.method == "tfidf_weighted":
            result_df = self._weighted_pool_single(
                comment_words_df,
                words_col="comment_words_filtered",
                out_col="comment_w2v_tfidf",
                vocab_df=self.comment_vocab_df,
                idf_df=self.comment_idf_df,
                vector_df=self.comment_vector_df,
            )
        else:
            result_df = self._max_pool_single(
                comment_words_df,
                words_col="comment_words_filtered",
                out_col="comment_w2v_max",
                vector_df=self.comment_vector_df,
            )

        self._assert_integrity(
            result_df.select("id"),
            f"{dataset_name}-comment-stage-output",
            expected_count=self._get_expected_count(),
            check_patterns=False,
            mode="light",
        )

        result_df = self._persist_df(result_df, f"{dataset_name}-comment-stage-output-memory")
        print("[ok] comment stage kept in memory")
        return result_df

    def assemble_improved_text_only(self, title_df, comment_df):
        dataset_name = self._get_dataset_name()

        self._assert_integrity(
            title_df.select("id"),
            f"{dataset_name}-assemble-title-input",
            expected_count=self._get_expected_count(),
            check_patterns=False,
            mode="light",
        )
        self._assert_integrity(
            comment_df.select("id"),
            f"{dataset_name}-assemble-comment-input",
            expected_count=self._get_expected_count(),
            check_patterns=False,
            mode="light",
        )

        improved_df = title_df.join(comment_df, on="id", how="inner")

        self._assert_integrity(
            improved_df.select("id"),
            f"{dataset_name}-improve-text-output",
            expected_count=self._get_expected_count(),
            check_patterns=False,
            mode="full",
        )

        improved_df = self._persist_df(improved_df, f"{dataset_name}-improved-text-memory")
        print("[ok] improved text features kept in memory")
        return improved_df

    # -------------------- main build flow --------------------
    def improve_text_features(self, method=None):
        if method is not None:
            self.method = method

        words_path = self.prepare_words_stage()
        words_df = self.spark.read.parquet(str(words_path)).select("id", "title_words_filtered", "comment_words_filtered")
        self._assert_integrity(
            words_df.select("id"),
            f"{self._get_dataset_name()}-shared-words-memory",
            expected_count=self._get_expected_count(),
            check_patterns=False,
            mode="light",
        )
        words_df = self._persist_df(words_df, f"{self._get_dataset_name()}-shared-words-memory")

        title_df = self.build_title_feature_only(words_df)
        comment_df = self.build_comment_feature_only(words_df)
        improved_df = self.assemble_improved_text_only(title_df, comment_df)

        self.cleanup_stage_resources(words_df, title_df, comment_df)

        print("result_df.sample(3):")
        improved_df.limit(3).show(truncate=False)
        return improved_df

    def save_improved_features(self, improved_df):
        if self.method == "tfidf_weighted":
            self.new_cols = ["title_w2v_tfidf", "comment_w2v_tfidf"]
        elif self.method == "max_pooling":
            self.new_cols = ["title_w2v_max", "comment_w2v_max"]
        else:
            raise ValueError(f"Unknown method: {self.method}")

        self._assert_integrity(
            improved_df.select("id"),
            f"{self._get_dataset_name()}-save-improved-features",
            expected_count=self._get_expected_count(),
            check_patterns=False,
            mode="light",
        )

        if self.output_path:
            self.output_path.parent.mkdir(parents=True, exist_ok=True)
            improved_df.coalesce(8).write.mode("overwrite").parquet(str(self.output_path))
            print(f"[ok] improved text features compatibility snapshot saved to: {self.output_path}")
        else:
            print("[ok] improved text features confirmed in memory")

        return improved_df


    def improve_dataset(self):
        print(f"=== step 1: build improved text features ({self._get_dataset_name()}) ===")

        self.title_w2v, self.comment_w2v = self.load_word2vec_model()
        self.vector_size = self.title_w2v.getVectorSize()

        self.title_vector_df = self._build_word_vector_df(self.title_w2v)
        self.comment_vector_df = self._build_word_vector_df(self.comment_w2v)

        if self.method == "tfidf_weighted":
            (title_cv, title_idf), (comment_cv, comment_idf) = self.load_tfidf_models()
            print(">>> build tfidf lookup tables")
            self.title_vocab_df = self._build_vocab_df(title_cv)
            self.comment_vocab_df = self._build_vocab_df(comment_cv)
            self.title_idf_df = self._build_idf_df(title_idf)
            self.comment_idf_df = self._build_idf_df(comment_idf)
            print(
                f"[ok] tfidf lookup ready: title_vocab={len(title_cv.vocabulary)}, "
                f"comment_vocab={len(comment_cv.vocabulary)}"
            )

        improved_df = self.improve_text_features(method=self.method)
        improved_df = self.save_improved_features(improved_df)
        print("[ok] improved text feature build finished")
        return improved_df

    def improve_train(self):
        return self.improve_dataset()

    def merge_features(self, improved_text_df):
        print(f"=== step 2: merge features ({self._get_dataset_name()}) ===")
        dataset_name = self._get_dataset_name()
        expected_count = self._get_expected_count()

        original_df = self.load_first_level_features()
        base_count, base_unique = self._assert_integrity(
            original_df,
            f"{dataset_name}-original-before-merge",
            expected_count=expected_count,
            check_patterns=True,
            mode="full",
        )

        self._assert_integrity(
            improved_text_df.select("id"),
            f"{dataset_name}-improved-text-before-merge",
            expected_count=expected_count,
            check_patterns=False,
            mode="light",
        )

        if self.method == "tfidf_weighted":
            self.new_cols = ["title_w2v_tfidf", "comment_w2v_tfidf"]
        else:
            self.new_cols = ["title_w2v_max", "comment_w2v_max"]

        cols_to_drop = [c for c in ["title_w2v_features", "comment_w2v_features"] if c in original_df.columns]
        remaining_df = original_df.drop(*cols_to_drop)

        final_df = remaining_df.join(
            improved_text_df.select("id", *self.new_cols),
            on="id",
            how="left",
        )

        merged_count, merged_unique = self._assert_integrity(
            final_df,
            f"{dataset_name}-after-merge",
            expected_count=expected_count,
            check_patterns=True,
            mode="full",
        )

        if merged_count != base_count or merged_unique != base_unique:
            raise ValueError(
                f"{dataset_name} merge size changed: "
                f"base total={base_count}, base unique={base_unique}, "
                f"merged total={merged_count}, merged unique={merged_unique}"
            )

        sample_ids = [148, 229, 307, 326, 463] if dataset_name == "train" else [1, 2, 3, 4, 5]
        compare_cols = ["user_id", "prod_id", "time", "votes", "purchased"]
        if dataset_name == "train":
            compare_cols.append("rating")
        self._safe_sample_compare(original_df, final_df, sample_ids, compare_cols)

        final_output_path = self._resolve_final_output_path()
        final_output_path.parent.mkdir(parents=True, exist_ok=True)
        final_df.repartition(8).write.mode("overwrite").parquet(str(final_output_path))

        print(f"[ok] final merged parquet saved to: {final_output_path}")
        self.final_improved_path = final_output_path

        self.cleanup_stage_resources(original_df, remaining_df, final_df)

    # -------------------- evaluation helpers --------------------
    def _load_eval_word_vectors(self, words):
        target_words = sorted(set(words))
        if not target_words:
            return

        missing_words = [w for w in target_words if w not in self.word_vectors]
        if not missing_words:
            return

        title_model, comment_model = self.load_word2vec_model()
        self.vector_size = title_model.getVectorSize()

        title_rows = title_model.getVectors().filter(F.col("word").isin(missing_words)).collect()
        comment_rows = comment_model.getVectors().filter(F.col("word").isin(missing_words)).collect()

        for row in title_rows:
            self.word_vectors[row["word"]] = row["vector"].toArray()
        for row in comment_rows:
            self.word_vectors[row["word"]] = row["vector"].toArray()

        print(f"[ok] eval word vectors loaded: {len(missing_words)} new words")

    def verify_sentiment_retention(self, df, original_col, improved_col, sample_frac=0.1):
        import pandas as pd
        import numpy as np

        sample_df = df.sample(fraction=sample_frac, seed=42).limit(10000)
        sample_pd = sample_df.toPandas()

        sample_pd["rating"] = pd.to_numeric(sample_pd["rating"], errors="coerce")
        sample_pd = sample_pd.dropna(subset=["rating", original_col, improved_col]).reset_index(drop=True)
        print(f"valid sampled rows: {len(sample_pd)}")

        if len(sample_pd) == 0:
            return {
                "acc_original": 0.0,
                "acc_improved": 0.0,
                "corr_original": 0.0,
                "corr_improved": 0.0,
                "std_norm_original": 0.0,
                "std_norm_improved": 0.0,
            }

        pos_words = [
            "great", "good", "perfect", "excellent", "awesome", "amazing", "love", "nice",
            "wonderful", "fantastic", "best", "satisfied", "happy", "impressive", "sturdy",
            "durable", "strong", "reliable", "versatile", "comfortable", "powerful", "clean",
            "easy", "quick", "fast", "worth", "value", "quality", "recommend", "works", "worked",
            "working", "well", "fit", "super", "top", "solid", "high", "efficient", "effective",
            "handy", "useful", "helpful", "glad", "pleased", "highly", "favorite", "beautiful",
            "cute", "fun", "cool", "bright", "safe", "decent", "fair", "okay", "ok", "fine",
        ]
        neg_words = [
            "bad", "poor", "terrible", "awful", "horrible", "disappointing", "disappointed",
            "junk", "garbage", "waste", "cheap", "flimsy", "weak", "useless", "worthless",
            "broken", "defective", "wrong", "missing", "leak", "stuck", "loose", "stopped",
            "damage", "damaged", "poorly", "cheaply", "pricey", "expensive", "hard", "difficult",
            "issue", "problem", "problems", "never", "nothing", "zero", "dead", "fail", "failed",
            "sucks", "horrible", "awful", "terrible", "inaccurate", "incorrect", "false", "misleading", "wrong",
        ]

        self._load_eval_word_vectors(pos_words + neg_words)

        acc_orig = self._classification_accuracy(sample_pd, original_col)
        corr_orig = self._sentiment_projection_correlation(sample_pd, original_col, pos_words, neg_words)
        std_orig = self._pooling_variance(sample_pd, original_col)

        acc_imp = self._classification_accuracy(sample_pd, improved_col)
        corr_imp = self._sentiment_projection_correlation(sample_pd, improved_col, pos_words, neg_words)
        std_imp = self._pooling_variance(sample_pd, improved_col)

        report = {
            "acc_original": acc_orig,
            "acc_improved": acc_imp,
            "corr_original": corr_orig,
            "corr_improved": corr_imp,
            "std_norm_original": std_orig,
            "std_norm_improved": std_imp,
        }

        print("\n=== sentiment retention report ===")
        print(f"accuracy: base={acc_orig:.4f}, improved={acc_imp:.4f}")
        print(f"corr:     base={corr_orig:.4f}, improved={corr_imp:.4f}")
        print(f"stdnorm:  base={std_orig:.4f}, improved={std_imp:.4f}")
        return report

    def _classification_accuracy(self, df_pd, vec_col, test_size=0.2):
        import numpy as np
        from sklearn.model_selection import train_test_split
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import accuracy_score

        X = np.array([v.toArray() for v in df_pd[vec_col].values])
        y = (df_pd["rating"] > 3.1).astype(int).values

        if len(np.unique(y)) < 2:
            return 0.0

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
        model = LogisticRegression(max_iter=1000)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        return accuracy_score(y_test, y_pred)

    def _sentiment_projection_correlation(self, df_pd, vec_col, pos_words, neg_words):
        import numpy as np
        from scipy.stats import pearsonr

        vectors = np.array([v.toArray() for v in df_pd[vec_col].values])

        pos_vecs = [self.word_vectors.get(w) for w in pos_words if w in self.word_vectors]
        neg_vecs = [self.word_vectors.get(w) for w in neg_words if w in self.word_vectors]
        if not pos_vecs or not neg_vecs:
            return 0.0

        pos_mean = np.mean(pos_vecs, axis=0)
        neg_mean = np.mean(neg_vecs, axis=0)
        direction = pos_mean - neg_mean

        if np.linalg.norm(direction) == 0:
            return 0.0

        direction = direction / np.linalg.norm(direction)
        projections = np.dot(vectors, direction)

        if np.std(projections) == 0:
            return 0.0

        corr, _ = pearsonr(projections, df_pd["rating"].values)
        return corr

    def _pooling_variance(self, df_pd, vec_col):
        import numpy as np
        vectors = np.array([v.toArray() for v in df_pd[vec_col].values])
        norms = np.linalg.norm(vectors, axis=1)
        return np.std(norms)

    def generate_verification_report(self, eval_df, original_col, improved_col):
        report = self.verify_sentiment_retention(eval_df, original_col, improved_col)
        report_path = self._resolve_output_dir() / "text_improvement_report.txt"

        with open(report_path, "w", encoding="utf-8") as f:
            f.write("=== text improvement verification report ===\n")
            f.write(f"accuracy_base={report['acc_original']:.4f}\n")
            f.write(f"accuracy_improved={report['acc_improved']:.4f}\n")
            f.write(f"corr_base={report['corr_original']:.4f}\n")
            f.write(f"corr_improved={report['corr_improved']:.4f}\n")
            f.write(f"std_base={report['std_norm_original']:.4f}\n")
            f.write(f"std_improved={report['std_norm_improved']:.4f}\n")

        print(f"[ok] verification report saved to: {report_path}")

    def verify_text(self):
        print("=== verify text features ===")

        if self._get_dataset_name() != "train":
            print("skip verify_text() because current dataset is test")
            return

        final_path = self.final_improved_path or self._resolve_final_output_path()

        original_df = self.spark.read.parquet(str(self.first_level_data_path))
        improved_df = self.spark.read.parquet(str(final_path))

        self._assert_integrity(
            improved_df,
            f"{self._get_dataset_name()}-verify-input",
            expected_count=self._get_expected_count(),
            check_patterns=True,
            mode="full",
        )

        original_vec_col = "title_w2v_features"
        improved_vec_col = "title_w2v_tfidf" if self.method == "tfidf_weighted" else "title_w2v_max"

        eval_df = original_df.select("id", "rating", original_vec_col).join(
            improved_df.select("id", improved_vec_col),
            on="id",
            how="inner",
        )

        self.generate_verification_report(eval_df, original_vec_col, improved_vec_col)
        self.cleanup_stage_resources(original_df, improved_df, eval_df)

    def run(self):
        final_output_path = self._resolve_final_output_path()
        if final_output_path.exists() and not self.force_recompute:
            print(f">>> skip existing final output: {final_output_path}")
            self.final_improved_path = final_output_path
            return

        improved_text_df = self.improve_dataset()
        self.merge_features(improved_text_df)
        self.cleanup_stage_resources(improved_text_df)
        print("[ok] main run finished; call verify_text() separately if needed")


In [ ]:
weighted_train = TextFeatures(
    spark=spark,
    raw_data_path=str(TRAIN_PATH),
    first_level_data_path=str(FEATURE_DIR / "final_train.parquet"),
    w2v_model_path=MODEL_DIR,
    tfidf_model_path=MODEL_DIR,
    output_path=str(FEATURE_DIR / "improved_text_only_train_weighted.parquet"),
    method="tfidf_weighted",
    force_recompute=FORCE_RECOMPUTE_FEATURES,
)

weighted_train.output_dir = FEATURE_DIR
weighted_train.expected_count = EXPECTED_TRAIN_ROWS if "EXPECTED_TRAIN_ROWS" in globals() else 5_173_005
weighted_train.dataset_name = "train"

print(">>> 开始执行 weighted_tfidf 训练集主流程...")
weighted_train.run()

In [ ]:
import gc

print(">>> 开始清理 weighted_train 后的 Python / Spark / scratch 资源...")

for var_name in [
    "weighted_train",
    "weighted_verifier",
    "compare_helper",
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

try:
    spark.catalog.clearCache()
except Exception:
    pass

try:
    spark.stop()
except Exception:
    pass

gc.collect()

# 保留 shared_words，避免 weighted_test 重跑分词
for p in SPARK_LOCAL_DIRS:
    clear_dir_contents(p)

clear_dir_contents(METHOD_STAGE_ROOT)

print(">>> 正在重建 SparkSession...")
spark = build_spark_session()

print("✅ Spark 已重启完成，开始跑 weighted_tfidf 测试集。")
print(f"SPARK_LOCAL_DIRS_VALUE: {SPARK_LOCAL_DIRS_VALUE}")
for p in SPARK_LOCAL_DIRS:
    print(f" - {p}")

weighted_test = TextFeatures(
    spark=spark,
    raw_data_path=str(TEST_PATH),
    first_level_data_path=str(FEATURE_DIR / "final_test.parquet"),
    w2v_model_path=MODEL_DIR,
    tfidf_model_path=MODEL_DIR,
    output_path=str(FEATURE_DIR / "improved_text_only_test_weighted.parquet"),
    method="tfidf_weighted",
    force_recompute=FORCE_RECOMPUTE_FEATURES,
)

weighted_test.output_dir = FEATURE_DIR
weighted_test.expected_count = EXPECTED_TEST_ROWS if "EXPECTED_TEST_ROWS" in globals() else None
weighted_test.dataset_name = "test"

print(">>> 开始执行 weighted_tfidf 测试集主流程...")
weighted_test.run()


In [ ]:
import gc

print(">>> 开始清理 weighted_test 后的 Python / Spark / scratch 资源...")

for var_name in [
    "weighted_test",
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

try:
    spark.catalog.clearCache()
except Exception:
    pass

try:
    spark.stop()
except Exception:
    pass

gc.collect()

# 保留 shared_words，避免 max_pooling 重跑分词
for p in SPARK_LOCAL_DIRS:
    clear_dir_contents(p)

clear_dir_contents(METHOD_STAGE_ROOT)

print(">>> 正在重建 SparkSession...")
spark = build_spark_session()

print("✅ Spark 已重启完成，可以开始跑 max_pooling 训练集。")
print(f"SPARK_LOCAL_DIRS_VALUE: {SPARK_LOCAL_DIRS_VALUE}")
for p in SPARK_LOCAL_DIRS:
    print(f" - {p}")


In [ ]:
max_output_dir = VERSION_DIR / "featured_data_max_pooling"
max_output_dir.mkdir(parents=True, exist_ok=True)

max_train = TextFeatures(
    spark=spark,
    raw_data_path=str(TRAIN_PATH),
    first_level_data_path=str(FEATURE_DIR / "final_train.parquet"),
    w2v_model_path=MODEL_DIR,
    tfidf_model_path=MODEL_DIR,
    output_path=str(max_output_dir / "improved_text_only_train_max.parquet"),
    method="max_pooling",
    force_recompute=FORCE_RECOMPUTE_FEATURES,
)

max_train.output_dir = max_output_dir
max_train.expected_count = EXPECTED_TRAIN_ROWS if "EXPECTED_TRAIN_ROWS" in globals() else 5_173_005
max_train.dataset_name = "train"

print(">>> 开始执行 max_pooling 训练集主流程...")
max_train.run()



In [ ]:
import gc

print(">>> 开始清理 max_train 后的 Python / Spark / scratch 资源...")

for var_name in [
    "max_train",
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

try:
    spark.catalog.clearCache()
except Exception:
    pass

try:
    spark.stop()
except Exception:
    pass

gc.collect()

# 保留 shared_words，避免 max_test 重跑分词
for p in SPARK_LOCAL_DIRS:
    clear_dir_contents(p)

clear_dir_contents(METHOD_STAGE_ROOT)

print(">>> 正在重建 SparkSession...")
spark = build_spark_session()

print("✅ Spark 已重启完成，开始跑 max_pooling 测试集。")
print(f"SPARK_LOCAL_DIRS_VALUE: {SPARK_LOCAL_DIRS_VALUE}")
for p in SPARK_LOCAL_DIRS:
    print(f" - {p}")

max_output_dir = VERSION_DIR / "featured_data_max_pooling"
max_output_dir.mkdir(parents=True, exist_ok=True)

max_test = TextFeatures(
    spark=spark,
    raw_data_path=str(TEST_PATH),
    first_level_data_path=str(FEATURE_DIR / "final_test.parquet"),
    w2v_model_path=MODEL_DIR,
    tfidf_model_path=MODEL_DIR,
    output_path=str(max_output_dir / "improved_text_only_test_max.parquet"),
    method="max_pooling",
    force_recompute=FORCE_RECOMPUTE_FEATURES,
)

max_test.output_dir = max_output_dir
max_test.expected_count = EXPECTED_TEST_ROWS if "EXPECTED_TEST_ROWS" in globals() else None
max_test.dataset_name = "test"

print(">>> 开始执行 max_pooling 测试集主流程...")
max_test.run()


In [ ]:
weighted_verifier = TextFeatures(
    spark=spark,
    raw_data_path=str(TRAIN_PATH),
    first_level_data_path=str(FEATURE_DIR / "final_train.parquet"),
    w2v_model_path=MODEL_DIR,
    tfidf_model_path=MODEL_DIR,
    output_path=str(FEATURE_DIR / "improved_text_only_train_weighted.parquet"),
    method="tfidf_weighted",
    force_recompute=False,
)

weighted_verifier.output_dir = FEATURE_DIR
weighted_verifier.expected_count = EXPECTED_TRAIN_ROWS if "EXPECTED_TRAIN_ROWS" in globals() else 5_173_005
weighted_verifier.dataset_name = "train"
weighted_verifier.final_improved_path = FEATURE_DIR / "final_improved_train.parquet"

print(">>> 开始验证 weighted_tfidf 训练集...")
weighted_verifier.verify_text()


In [ ]:
max_output_dir = VERSION_DIR / "featured_data_max_pooling"

max_verifier = TextFeatures(
    spark=spark,
    raw_data_path=str(TRAIN_PATH),
    first_level_data_path=str(FEATURE_DIR / "final_train.parquet"),
    w2v_model_path=MODEL_DIR,
    tfidf_model_path=MODEL_DIR,
    output_path=str(max_output_dir / "improved_text_only_train_max.parquet"),
    method="max_pooling",
    force_recompute=False,
)

max_verifier.output_dir = max_output_dir
max_verifier.expected_count = EXPECTED_TRAIN_ROWS if "EXPECTED_TRAIN_ROWS" in globals() else 5_173_005
max_verifier.dataset_name = "train"
max_verifier.final_improved_path = max_output_dir / "final_improved_train.parquet"

print(">>> 开始验证 max_pooling 训练集...")
max_verifier.verify_text()


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

my_pos_words = [
    "great", "good", "perfect", "excellent", "awesome", "amazing", "love", "nice",
    "wonderful", "fantastic", "best", "satisfied", "happy", "impressive", "sturdy",
    "durable", "strong", "reliable", "versatile", "comfortable", "powerful", "clean",
    "easy", "quick", "fast", "worth", "value", "quality", "recommend", "works", "worked",
    "working", "well", "fit", "super", "top", "solid", "high", "efficient", "effective",
    "handy", "useful", "helpful", "glad", "pleased", "highly", "favorite", "beautiful",
    "cute", "fun", "cool", "bright", "safe", "decent", "fair", "okay", "ok", "fine"
]

my_neg_words = [
    "bad", "poor", "terrible", "awful", "horrible", "disappointing", "disappointed",
    "junk", "garbage", "waste", "cheap", "flimsy", "weak", "useless", "worthless",
    "broken", "defective", "wrong", "missing", "leak", "stuck", "loose", "stopped",
    "damage", "damaged", "poorly", "cheaply", "pricey", "expensive", "hard", "difficult",
    "issue", "problem", "problems", "never", "nothing", "zero", "dead", "fail", "failed",
    "sucks", "inaccurate", "incorrect", "false", "misleading", "wrong"
]

path_v1 = str(FEATURE_DIR / "final_train.parquet")
path_v2 = str((VERSION_DIR / "featured_data_max_pooling" / "final_improved_train.parquet"))
path_v3 = str(FEATURE_DIR / "final_improved_train.parquet")

print(">>> 正在加载 V1 / V2 / V3 结果进行统一对比...")

df_v1 = spark.read.parquet(path_v1).select("id", "rating", "title_w2v_features")
df_v2 = spark.read.parquet(path_v2).select("id", "title_w2v_max")
df_v3 = spark.read.parquet(path_v3).select("id", "title_w2v_tfidf")

eval_df = (
    df_v1.join(df_v2, on="id", how="inner")
         .join(df_v3, on="id", how="inner")
)

sample_pd = eval_df.sample(fraction=0.1, seed=42).limit(10000).toPandas()
sample_pd["rating"] = pd.to_numeric(sample_pd["rating"], errors="coerce")
sample_pd = sample_pd.dropna().reset_index(drop=True)

print(f">>> 有效采样数: {len(sample_pd)}")

compare_helper = TextFeatures(
    spark=spark,
    raw_data_path=str(TRAIN_PATH),
    first_level_data_path=str(FEATURE_DIR / "final_train.parquet"),
    w2v_model_path=MODEL_DIR,
    tfidf_model_path=MODEL_DIR,
    output_path=str(FEATURE_DIR / "improved_text_only_train_weighted.parquet"),
    method="tfidf_weighted",
    force_recompute=False,
)
compare_helper._load_eval_word_vectors(my_pos_words + my_neg_words)
w2v_dict = compare_helper.word_vectors


def get_metrics(df, vec_col, pos_list, neg_list, word_vec_dict):
    X = np.array([v.toArray() for v in df[vec_col].values])
    y = df["rating"].astype(float).values

    y_bin = (y > 3.1).astype(int)
    if len(np.unique(y_bin)) < 2:
        acc = 0.0
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y_bin, test_size=0.2, random_state=42
        )
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train, y_train)
        acc = accuracy_score(y_test, clf.predict(X_test))

    pos_vecs = [word_vec_dict.get(w) for w in pos_list if w in word_vec_dict]
    neg_vecs = [word_vec_dict.get(w) for w in neg_list if w in word_vec_dict]

    corr = 0.0
    if pos_vecs and neg_vecs:
        direction = np.mean(pos_vecs, axis=0) - np.mean(neg_vecs, axis=0)
        norm = np.linalg.norm(direction)
        if norm > 0:
            direction = direction / norm
            projections = np.dot(X, direction)
            if np.std(projections) > 0:
                corr, _ = pearsonr(projections, y)

    std_norm = np.std(np.linalg.norm(X, axis=1))
    return acc, corr, std_norm


results = []
for version_name, vec_col in [
    ("V1 (原始均值)", "title_w2v_features"),
    ("V2 (最大池化)", "title_w2v_max"),
    ("V3 (TF-IDF加权)", "title_w2v_tfidf"),
]:
    acc, corr, std_norm = get_metrics(sample_pd, vec_col, my_pos_words, my_neg_words, w2v_dict)
    results.append({
        "版本": version_name,
        "分类准确率 Acc ↑": f"{acc:.4f}",
        "情感相关性 Corr ↑": f"{corr:.4f}",
        "范数标准差 Std ↓": f"{std_norm:.4f}",
    })

result_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("文本特征统一对比报告")
print("=" * 70)
print(result_df.to_string(index=False))
print("=" * 70)

acc_v2 = float(result_df.loc[result_df["版本"] == "V2 (最大池化)", "分类准确率 Acc ↑"].iloc[0])
corr_v2 = float(result_df.loc[result_df["版本"] == "V2 (最大池化)", "情感相关性 Corr ↑"].iloc[0])
std_v2 = float(result_df.loc[result_df["版本"] == "V2 (最大池化)", "范数标准差 Std ↓"].iloc[0])

acc_v3 = float(result_df.loc[result_df["版本"] == "V3 (TF-IDF加权)", "分类准确率 Acc ↑"].iloc[0])
corr_v3 = float(result_df.loc[result_df["版本"] == "V3 (TF-IDF加权)", "情感相关性 Corr ↑"].iloc[0])
std_v3 = float(result_df.loc[result_df["版本"] == "V3 (TF-IDF加权)", "范数标准差 Std ↓"].iloc[0])

print("\n重点比较：TF-IDF 加权 vs 最大池化")
print(f"Acc 提升:  {acc_v3 - acc_v2:+.4f}")
print(f"Corr 提升: {corr_v3 - corr_v2:+.4f}")
print(f"Std 变化:  {std_v3 - std_v2:+.4f}")
